# Pre-requisites of LangGraph

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
required_keys = [
    "GROQ_API_KEY",
]

missing_keys = []

for key in required_keys:
    value = os.getenv(key)
    if value is None:
        missing_keys.append(key)
    else:
        os.environ[key] = value

if missing_keys:
    print(f"Missing environment variables: {missing_keys}")
    print("Add them in your .env file before running the notebook.")
else:
    print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
from langchain_groq import ChatGroq
chat = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

## Simple AI Assistant

In [4]:
while True:
    question = input("Type Your Question: If you want to quit the chat write quit")
    if question != "quit":
        print(chat.invoke(question).content)
    else:
        print("Goodbye, Take care of yourself!")
        break

Women empowerment refers to the process of enabling women to have control over their own lives, make informed decisions, and take action to improve their well-being and achieve their full potential. It involves creating an environment where women have equal access to resources, opportunities, and power, and can participate fully in all aspects of society.

**Key Aspects of Women Empowerment:**

1. **Education**: Providing access to quality education, vocational training, and literacy programs to equip women with the skills and knowledge they need to make informed decisions.
2. **Economic Empowerment**: Enabling women to participate in the formal economy, access credit and financial services, and own businesses or property.
3. **Health and Nutrition**: Ensuring women have access to quality healthcare, nutrition, and reproductive health services to improve their physical and mental well-being.
4. **Social and Cultural Change**: Addressing social and cultural norms that perpetuate inequal

In [5]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage

In [6]:
store={}

In [7]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [8]:
config = {"configurable": {"session_id": "firstchat"}}

In [9]:
model_with_memory = RunnableWithMessageHistory(
    chat,
    get_session_history=get_session_history,
)

In [10]:
model_with_memory.invoke(("Hi! I'm Sankalp Bankar"), config=config).content

"Hello Sankalp Bankar, it's nice to meet you. Is there something I can help you with or would you like to chat?"

In [11]:
model_with_memory.invoke(("Tell me what is my name?"), config=config).content

'Your name is Sankalp Bankar.'

In [12]:
store

{'firstchat': InMemoryChatMessageHistory(messages=[HumanMessage(content="Hi! I'm Sankalp Bankar", additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Sankalp Bankar, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 45, 'total_tokens': 76, 'completion_time': 0.037888709, 'prompt_time': 0.002161738, 'queue_time': 0.049260121, 'total_time': 0.040050447}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--516a8b0d-fa28-45a3-a4a3-4ac31a24857d-0', usage_metadata={'input_tokens': 45, 'output_tokens': 31, 'total_tokens': 76}), HumanMessage(content='Tell me what is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Sankalp Bankar.', additional_kwargs={}, response_meta

## RAG with LCEL

In [13]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [14]:
from langchain_community.document_loaders import TextLoader

# Use your correct local file path here
file_path = "playstation_evolution.txt"

loader = TextLoader(file_path, encoding="utf-8")
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")

Loaded 1 document(s)


In [15]:
# Creating Chunks using RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10, length_function=len)
new_docs = text_splitter.split_documents(docs)
doc_strings = [doc.page_content for doc in new_docs]

In [16]:
# Creating Retriever using Vector DB
db = Chroma.from_texts(doc_strings, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

In [17]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

In [18]:
# Creating RAG Chain using LCEL
retrieval_chain = (
    RunnableParallel(
        context=(retriever | (lambda docs: "\n".join([doc.page_content for doc in docs]))),
        question=RunnablePassthrough()
    )
    | prompt
    | chat
    | StrOutputParser()
)

In [19]:
question = "How was PlayStation Evolution?"
retrieval_chain.invoke(question)

'The PlayStation brought a revolution in gaming, transforming it from a simple hobby into a global entertainment phenomenon.'

## Tools and Agents

In [20]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [21]:
api_wrapper = WikipediaAPIWrapper()

In [22]:
tool = WikipediaQueryRun(api_wrapper = api_wrapper)

In [23]:
tool.name

'wikipedia'

In [24]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [25]:
tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [26]:
print(tool.run("playstation"))

Page: PlayStation
Summary: PlayStation is a video gaming brand owned and produced by Sony Interactive Entertainment (SIE), a subsidiary of Japanese conglomerate Sony. Its flagship products consist of a series of home video game consoles produced under the brand; it also consists of handhelds, online services, magazines, and other forms of media.
The brainchild of Sony executive Ken Kutaragi, the brand began with the first PlayStation home console released in Japan in 1994 and worldwide the following year, which became the first console of any type to ship over 100 million units, which made PlayStation a globally recognized brand. Since then there have been numerous newer consoles—the most recent being the PlayStation 5 released in 2020—while there have also been a series of handheld consoles and a number of other electronics such as a media center and a smartphone. The main series of controllers utilized by the PlayStation series is the DualShock, a line of vibration-feedback gamepads.

In [27]:
from youtubesearchpython import VideosSearch

class YoutubeSearchTool:
    def __init__(self, max_results=5):
        self.max_results = max_results
        self.name = "YoutubeSearchToolFallback"

    def run(self, query):
        videos = VideosSearch(query, limit=self.max_results)
        return videos.result()

In [28]:
# Initialize fallback Youtube search tool
tool2 = YoutubeSearchTool(max_results=5)
print('Initialized:', tool2.name)

Initialized: YoutubeSearchToolFallback


In [29]:
# Show tool metadata
tool2.name

'YoutubeSearchToolFallback'

In [30]:
results = tool2.run("Sankalp Bankar")

for item in results.get("result", []):
    print(item.get("title"))
    print(item.get("link"))
    print("-" * 50)

TypeError: post() got an unexpected keyword argument 'proxies'

In [31]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [32]:
from langchain_community.tools.tavily_search import TavilySearchResults
import os

# Make sure TAVILY_API_KEY exists in your .env file
tool3 = TavilySearchResults(
    tavily_api_key=os.getenv("TAVILY_API_KEY")
)

C:\Users\LOQ\AppData\Local\Temp\ipykernel_31352\1010966577.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool3 = TavilySearchResults(


In [33]:
tool3.invoke("What is the capital of France?")

[{'title': 'Paris, France | Geography and Cartography | Research Starters',
  'url': 'https://www.ebsco.com/research-starters/geography-and-cartography/paris-france',
  'content': 'Browse Subject Areas\n\n# Paris, France\n\nParis, the capital of France, is renowned globally for its iconic landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. This vibrant city, located along the Seine River in the Île de France region, is divided into 20 arrondissements, each with its unique character and cultural significance. Paris is not only a historical epicenter but also a major hub for international fashion, attracting millions of tourists each year—over 40 million in 2022—who come to experience its art, cuisine, and diverse attractions. [...] ## Paris, France\n\nParis is the capital of France and is known throughout the world as the home of such attractions as the Louvre Museum, the Eiffel Tower, and Notre-Dame Cathedral. It is also internationally recognized as a cent

In [37]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Users\LOQ\anaconda3\Lib\site-packages\langchain\agents\__init__.py)

In [38]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

tools = [wikipedia_tool]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that uses tools to answer questions."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(chat, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Users\LOQ\anaconda3\Lib\site-packages\langchain\agents\__init__.py)

In [52]:
# Run agent with a query
result = agent_executor.invoke({"input": "What is the capital of Italy?"})

NameError: name 'agent_executor' is not defined

In [ ]:
print(result["output"])

NameError: name 'agent' is not defined